In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()

print(df.shape)
print(df.isna().sum())        # which columns have missing data?
print(df.head())

In [ ]:
import pandas as pd

# 1. fill missing ages with the median
df["age"] = df["age"].fillna(df["age"].median())
# 2. embarked is only missing in 2 rows → just drop them
df = df.dropna()
# 3. text columns → 0/1 columns (one-hot)
#    sex → sex_female, sex_male | embarked → embarked_C, embarked_Q, embarked_S
df = pd.get_dummies(df, columns=["sex", "embarked"], dtype=float)

FEATURES = [k for k in df.columns if k != "survived"]
print(FEATURES)
print(df.head())

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["Died", "Survived"]

X = df[FEATURES].values
y = df["survived"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def to_tensor(X, y):
    return TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))

train_loader = DataLoader(to_tensor(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(to_tensor(X_test, y_test), batch_size=32)

model = nn.Sequential(
    nn.Linear(len(FEATURES), 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 2),
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 50

# training
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} | loss {total_loss / len(train_loader):.4f}")